# CVA validation map

This notebook maps the public `frtb-cva` calculation entrypoints to the package validation fixtures and runs representative BA-CVA and SA-CVA fixture cases.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None

for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_cva").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    package_candidate = candidate / "packages" / "frtb-cva"
    if (package_candidate / "src" / "frtb_cva").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = package_candidate
        break

if PACKAGE_ROOT is None or REPO_ROOT is None:
    raise RuntimeError("Could not locate the frtb-cva package root")

workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src")))
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


PACKAGE_ROOT

In [ ]:
from cva_notebook_data import load_ba_fixture, load_sa_fixture, markdown_table
from frtb_cva import calculate_cva_capital, validate_cva_result_reconciliation

ba_fixture = load_ba_fixture()
sa_fixture = load_sa_fixture()

display(
    Markdown(
        markdown_table(
            ("fixture", "method", "valid cases", "invalid cases"),
            (
                (
                    ba_fixture.fixture_name,
                    ba_fixture.context.method.value,
                    len(ba_fixture.cases),
                    len(ba_fixture.invalid_cases),
                ),
                (
                    sa_fixture.fixture_name,
                    sa_fixture.context.method.value,
                    len(sa_fixture.cases),
                    len(sa_fixture.invalid_cases),
                ),
            ),
        )
    )
)

In [ ]:
ba_case_id, counterparties, netting_sets = ba_fixture.cases[0]
ba_result = calculate_cva_capital(ba_fixture.context, counterparties, netting_sets)
validate_cva_result_reconciliation(ba_result)

sa_case_id, sensitivities, hedges = sa_fixture.cases[0]
sa_result = calculate_cva_capital(
    sa_fixture.context,
    (),
    (),
    sensitivities=sensitivities,
    hedges=hedges,
)
validate_cva_result_reconciliation(sa_result)

display(
    Markdown(
        markdown_table(
            ("case", "total CVA capital", "result branches", "citations"),
            (
                (
                    ba_case_id,
                    f"{ba_result.total_cva_capital:,.2f}",
                    len(ba_result.ba_cva_counterparty_capitals),
                    ", ".join(ba_result.citations[:4]),
                ),
                (
                    sa_case_id,
                    f"{sa_result.total_cva_capital:,.2f}",
                    len(sa_result.sa_cva_risk_class_capitals),
                    ", ".join(sa_result.citations[:4]),
                ),
            ),
        )
    )
)

In [ ]:
invalid_rows = tuple(
    (fixture_name, case_id, expected)
    for fixture_name, invalid_cases in (
        (ba_fixture.fixture_name, ba_fixture.invalid_cases),
        (sa_fixture.fixture_name, sa_fixture.invalid_cases),
    )
    for case_id, expected, *_ in invalid_cases
)

display(Markdown(markdown_table(("fixture", "invalid case", "expected error"), invalid_rows)))